In [ ]:
!pip install transformers datasets evaluate torch pandas numpy

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig, TrainingArguments, Trainer
from datasets import Dataset
import evaluate
import numpy as np
import torch
import pandas as pd
import os
import json

In [ ]:
MODEL_CKPT = os.getenv("MODEL_CKPT", "distilbert/distilbert-base-uncased")
os.makedirs("../model", exist_ok=True)

label2id = {'Informative': 0, 'Misinformative': 1}
id2label = {0: 'Informative', 1: 'Misinformative'}

num_labels = len(label2id)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def get_data(file_path):
    df = pd.read_csv(file_path)
    return df

def tokenize_function(examples):
    return tokenizer(examples["title"], padding="max_length", truncation=True, max_length=512)

def compute_metrics(eval_pred):
    metric = evaluate.load("accuracy")
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
with open("../config.json", "r") as f:
    all_configs = json.load(f)

args_keys = [key for key in all_configs.keys() if key.startswith("args")]

In [ ]:
for args_key in args_keys:
    
    config_args = all_configs[args_key]
    
    config = AutoConfig.from_pretrained(
        MODEL_CKPT,
        label2id=label2id,
        id2label=id2label,
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_CKPT, config=config).to(device)

    train_df = get_data("../results/train.csv")
    val_df = get_data("../results/val.csv")

    train_dataset = Dataset.from_pandas(train_df)
    val_dataset = Dataset.from_pandas(val_df)

    train_dataset = train_dataset.map(tokenize_function, batched=True)
    val_dataset = val_dataset.map(tokenize_function, batched=True)

    train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
    val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

    training_args = TrainingArguments(
        output_dir=config_args["output_dir"],
        num_train_epochs=config_args["num_train_epochs"],
        learning_rate=config_args["learning_rate"],
        per_device_train_batch_size=config_args["per_device_train_batch_size"],
        per_device_eval_batch_size=config_args["per_device_eval_batch_size"],
        weight_decay=config_args["weight_decay"],
        disable_tqdm=config_args["disable_tqdm"],
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        compute_metrics=compute_metrics, 
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
    )

    print(model.config)

    trainer.train()
    
    del model, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()